# autoresearch-xla: High-MFU Resumable `1b-ve` FSDP Pretraining (Colab & Kaggle GPU/TPU)

Train the **Round-5 Champion `1b-ve` LLM** (**1.456B total / 1.120B active**, `20L × 2048D`, SwiGLU, GQA `16:4`, `SSSL` sliding window, U-Net skips, `ve_stride=2` full VE gate, elementwise attention gate, per-channel smear, Canon-A/C, `v0_residual`, `qk_gain`, **Qwen 3.8 Split-Muon ON**) in pure JAX/XLA:
- **True 1D FSDP Sharding**: Shards all weights & Split-Muon/AdamW states across all 8 TPU chips on Kaggle `v5e-8` (~364 MB/chip BF16+SR params + ~482 MB/chip BF16 opt = **< 1 GB/chip static HBM!**).
- **Full 3,152-TOPS INT8 + Pallas Stack**: Pallas TPU `SplashAttention(1024)` (with automatic 4-tier Kaggle `libtpu` self-healing unlock!), GPU `cuDNN FlashAttention`, single-pass FSDP **Fused Cut-Cross-Entropy** with INT8 backward `lm_head`, pure-BF16+SR master weights, `--matmul-quant int8-all`, all 10 verified MaxText `v5e-8` FSDP flags, and async H2D double-buffering.
- **Stage-1 Preflight → Auto-Promote**: Runs a fast ~2 min microbenchmark & HBM/loss/throughput gate before immediately launching 8.2h continuous training — zero wasted Kaggle TPU queues!
- **Atomic Private HuggingFace Hub Checkpointing**: Single-commit periodic uploads (`--save-every 500`, 3× retry resilience, `auto/autoresearch-1b-ve` repo auto-discovery, `--keep-last-checkpoints 1` on Kaggle's 19.5 GB disk) + WSD multi-session auto-resume!


## 1. Environment Setup & Self-Healing Kaggle TPU Pallas Unlock


In [ ]:
import ast
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV_NAME = "colab" if IN_COLAB else ("kaggle" if IN_KAGGLE else "local")
print(f"Environment: {ENV_NAME}", flush=True)

REPO_URL = "https://github.com/OttoApocalypse69/autoresearch-xla.git"
MOUNT_GOOGLE_DRIVE = True  # Only used on Colab

if Path("scale/pretrain.py").exists():
    REPO_DIR = Path.cwd()
else:
    # On Kaggle, clone into /tmp so /kaggle/working stays 100% free for checkpoints!
    target = Path("/content/autoresearch-xla" if IN_COLAB else ("/tmp/autoresearch-xla" if IN_KAGGLE else "./autoresearch-xla"))
    if (target / "scale" / "pretrain.py").exists():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        if target.exists():
            shutil.rmtree(target)
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(target)], check=True)
    os.chdir(target)
    REPO_DIR = target

DRIVE_DIR = None
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = "/content/drive/MyDrive/autoresearch-1b-ve-ckpt"
    except Exception as exc:
        print(f"Skipping Google Drive mount: {exc}", flush=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tiktoken", "pyarrow", "requests", "tokenizers", "huggingface-hub", "safetensors", "matplotlib"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rustbpe"], check=False)

# Self-healing Pallas TPU SplashAttention probe (runs in subprocess so Jupyter never holds /tmp/libtpu_lockfile!)
PROBE_PY = f"""
import json, os, sys
sys.path.insert(0, {str(REPO_DIR)!r})
out = {{"tpu_lib": os.environ.get("TPU_LIBRARY_PATH"), "libtpu_ver": None, "libtpu_path": None, "jax_ver": None, "n_dev": 0, "platform": None, "kind": None, "pallas_ok": False}}
try:
    import libtpu
    out["libtpu_ver"] = getattr(libtpu, "__version__", None)
    if hasattr(libtpu, "get_library_path"):
        out["libtpu_path"] = str(libtpu.get_library_path())
except Exception:
    pass
try:
    import jax
    out["jax_ver"] = jax.__version__
    devs = jax.devices()
    out["n_dev"] = len(devs)
    out["platform"] = str(devs[0].platform).lower() if devs else None
    out["kind"] = str(getattr(devs[0], "device_kind", "")) if devs else None
    if out["platform"] == "tpu":
        from scale.attention import _probe_pallas_tpu_supported
        out["pallas_ok"] = bool(_probe_pallas_tpu_supported())
except Exception as exc:
    out["error"] = f"{{type(exc).__name__}}: {{exc}}"
print("PROBE_JSON=" + json.dumps(out))
"""

def _probe(extra_env=None):
    env = {**os.environ, **(extra_env or {})}
    if extra_env and "TPU_LIBRARY_PATH" in extra_env and extra_env["TPU_LIBRARY_PATH"] is None:
        env.pop("TPU_LIBRARY_PATH", None)
    r = subprocess.run([sys.executable, "-c", PROBE_PY], cwd=str(REPO_DIR), env=env, capture_output=True, text=True)
    for ln in reversed(r.stdout.splitlines()):
        if ln.startswith("PROBE_JSON="):
            return json.loads(ln[len("PROBE_JSON="):])
    return {"platform": None, "pallas_ok": False, "n_dev": 0}

orig_tpu_lib = os.environ.get("TPU_LIBRARY_PATH")
HW_PROBE = _probe()
PALLAS_OK = bool(HW_PROBE.get("pallas_ok"))

if IN_KAGGLE and HW_PROBE.get("platform") == "tpu" and not PALLAS_OK:
    orig_jax = HW_PROBE.get("jax_ver")
    orig_lt = HW_PROBE.get("libtpu_ver")
    added_lt = False
    if HW_PROBE.get("libtpu_path") and HW_PROBE["libtpu_path"] != orig_tpu_lib:
        p1 = _probe({"TPU_LIBRARY_PATH": HW_PROBE["libtpu_path"]})
        if p1.get("pallas_ok"):
            os.environ["TPU_LIBRARY_PATH"] = HW_PROBE["libtpu_path"]
            HW_PROBE, PALLAS_OK = p1, True
    if not PALLAS_OK and orig_jax:
        if subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"jax[tpu]=={orig_jax}"]).returncode == 0:
            added_lt = True
            try:
                lp = subprocess.check_output([sys.executable, "-c", "import libtpu; print(libtpu.get_library_path())"], text=True).strip()
            except Exception:
                lp = None
            if lp:
                p2 = _probe({"TPU_LIBRARY_PATH": lp})
                if p2.get("pallas_ok"):
                    os.environ["TPU_LIBRARY_PATH"] = lp
                    HW_PROBE, PALLAS_OK = p2, True
    if not PALLAS_OK:
        if orig_tpu_lib:
            os.environ["TPU_LIBRARY_PATH"] = orig_tpu_lib
        else:
            os.environ.pop("TPU_LIBRARY_PATH", None)
        if added_lt and not orig_lt:
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "libtpu"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax==0.6.2", "jaxlib==0.6.2"], check=False)
        p3 = _probe({"TPU_LIBRARY_PATH": orig_tpu_lib})
        if p3.get("pallas_ok"):
            HW_PROBE, PALLAS_OK = p3, True

print(f"Repo ready at {REPO_DIR} | {HW_PROBE}", flush=True)


## 2. Inspect Hardware & FSDP Memory Plan (TPU-Lock-Safe Subprocess)


In [ ]:
import subprocess, sys

PRESET = "1b-ve"  # Round-5 Champion: 1.456B total / 1.120B active, Split-Muon ON, MTP OFF

code = f"""
import functools, jax, jax.numpy as jnp
from prepare import detect_hardware
from scale.config import get_model_preset
from scale.fsdp import create_fsdp_mesh
from scale.model import init_params, count_params
from scale.optimizer import init_opt_state

hw = detect_hardware()
spec = get_model_preset({PRESET!r})
abs_p = jax.eval_shape(functools.partial(init_params, spec=spec, param_dtype=jnp.bfloat16), jax.random.PRNGKey(0))
abs_o = jax.eval_shape(functools.partial(init_opt_state, opt_dtype=jnp.bfloat16, split_muon=True, spec=spec), abs_p)
pc = count_params(abs_p)
p_mb = sum(a.size * a.dtype.itemsize for a in jax.tree_util.tree_leaves(abs_p)) / 1024**2
o_mb = sum(a.size * a.dtype.itemsize for a in jax.tree_util.tree_leaves(abs_o)) / 1024**2
print(f"JAX {{jax.__version__}} | {{hw.device_kind}} x{{hw.num_devices}} ({{hw.platform}}, {{hw.compute_dtype}}, peak={{hw.peak_tflops_total:.1f}} TFLOPS)")
print(f"Preset {PRESET!r}: {{pc['total']/1e6:.2f}}M total / {{(pc['total']-pc['value_embeds'])/1e6:.2f}}M active ({{spec.n_layer}}L x {{spec.n_embd}}D, GQA {{spec.n_head}}:{{spec.n_kv_head}}, win={{spec.window_pattern}})")
print(f"BF16+SR state: params={{p_mb:.0f}} MB + opt={{o_mb:.0f}} MB = {{p_mb+o_mb:.0f}} MB total -> ~{{(p_mb+o_mb)/hw.num_devices:.0f}} MB/chip with FSDP (~{{(p_mb+o_mb)/1024:.2f}} GB/ckpt)")
"""
res = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Hardware probe failed.")


## 3. Preflight Gate -> Auto-Promote Resumable `1b-ve` FSDP Pretraining

- **Recommended persistence (`SYNC_BACKEND = "hf"`, `REMOTE_TARGET = "auto/autoresearch-1b-ve"`)**: Add `HF_TOKEN` to Kaggle Secrets / Colab Secrets. Automatically resolves `<your-hf-username>/autoresearch-1b-ve` as a **private** repo and uploads 1 atomic commit every 500 steps with 3× retry resilience!
- **Colab Google Drive persistence**: Automatically used on Colab if Drive is mounted and `HF_TOKEN` is unset.
- **Preflight Gate (`RUN_PREFLIGHT = True`)**: Verifies finite loss, HBM headroom, and throughput (~2 min) before immediately promoting to full training so you never waste a long Kaggle TPU queue!


In [ ]:
import math
import os
import re
import subprocess
import sys

# ---- User Configuration (Round-5 Champion 1b-ve) ----
PRESET = "1b-ve"
DATASET = "mix-1b"
TOTAL_TOKENS = 20_000_000_000
TOTAL_BATCH_SIZE = 524_288
PER_DEVICE_BATCH = 4              # 4 on v5e-8 / v6e-1 / A100 / H100; 2 on single 16GB v5e-1 / T4
PARAM_DTYPE = "bfloat16"
OPT_DTYPE = "bfloat16"
MATMUL_QUANT = "int8-all"         # 394 TOPS/chip INT8 MXU on TPU v5e ("none" for pure BF16, "fp8" on v6e/H100)
MUON_VMAP = True
REMAT_POLICY = "save_all_compute" if (not IN_KAGGLE or PALLAS_OK) else "save_attn"
ATTN_IMPL = "auto"
SPLASH_BLOCK_SIZE = 1024
LOSS_CHUNK_SIZE = 4096
MATRIX_LR = 0.04
EMBEDDING_LR = 0.5
WEIGHT_DECAY = 0.03

PHASE = "stable"                  # "stable" across 8.2h sessions, then "decay" for final ~2B cooldown!
WARMUP_STEPS = 500
DECAY_STEPS = 4000

RUN_PREFLIGHT = True              # Fast ~2 min sanity/speed gate before promoting immediately to full training
MAX_HOURS = 8.2 if IN_KAGGLE else 11.5
MAX_STEPS = None
SAVE_EVERY = 500
EVAL_EVERY = 250
SAMPLE_EVERY = 500
LOG_EVERY = 10
KEEP_LAST_CHECKPOINTS = 1 if IN_KAGGLE else 2
SHARD_CACHE_DIR = "/tmp/shards_mix1b"
CHECKPOINT_DIR = "/kaggle/working/ckpt_1b" if IN_KAGGLE else "./checkpoints/1b-ve"

# Discover HF_TOKEN from env / Colab / Kaggle Secrets (NEVER print!)
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
if "HF_TOKEN" not in env:
    for loader_fn in (
        lambda: __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN"),
        lambda: __import__("kaggle_secrets", fromlist=["UserSecretsClient"]).UserSecretsClient().get_secret("HF_TOKEN"),
    ):
        try:
            tok = loader_fn()
            if tok:
                env["HF_TOKEN"] = str(tok).strip()
                break
        except Exception:
            pass

SYNC_BACKEND = "hf" if env.get("HF_TOKEN") else ("local" if DRIVE_DIR else "none")
REMOTE_TARGET = "auto/autoresearch-1b-ve" if SYNC_BACKEND == "hf" else DRIVE_DIR
# -----------------------------------------------------

common_args = [
    "--preset", PRESET,
    "--total-batch-size", str(TOTAL_BATCH_SIZE),
    "--per-device-batch", str(PER_DEVICE_BATCH),
    "--param-dtype", PARAM_DTYPE,
    "--opt-dtype", OPT_DTYPE,
    "--matmul-quant", MATMUL_QUANT,
    "--remat-policy", REMAT_POLICY,
    "--attn-impl", ATTN_IMPL,
    "--splash-block-size", str(SPLASH_BLOCK_SIZE),
    "--loss-chunk-size", str(LOSS_CHUNK_SIZE),
    "--matrix-lr", str(MATRIX_LR),
    "--embedding-lr", str(EMBEDDING_LR),
    "--weight-decay", str(WEIGHT_DECAY),
]
if MUON_VMAP:
    common_args += ["--muon-vmap"]

if RUN_PREFLIGHT:
    print("=== Stage 1 Preflight Gate ===", flush=True)
    pf = subprocess.run(
        [sys.executable, "-m", "scale.pretrain", *common_args, "--benchmark-only", "--bench-steps", "6"],
        env=env, capture_output=True, text=True,
    )
    print(pf.stdout, flush=True)
    if pf.returncode != 0 and REMAT_POLICY == "save_all_compute":
        print("save_all_compute preflight failed — retrying with save_attn...", flush=True)
        REMAT_POLICY = "save_attn"
        common_args[common_args.index("--remat-policy") + 1] = "save_attn"
        pf = subprocess.run(
            [sys.executable, "-m", "scale.pretrain", *common_args, "--benchmark-only", "--bench-steps", "6"],
            env=env, capture_output=True, text=True,
        )
        print(pf.stdout, flush=True)
    if pf.returncode != 0:
        print(pf.stderr[-1000:], flush=True)
        raise RuntimeError(f"Preflight failed with code {pf.returncode}")
    m = re.search(r"loss=([^\s|]+)", pf.stdout)
    assert m and math.isfinite(float(m.group(1))) and float(m.group(1)) < 15.0, f"Preflight bad loss: {m}"
    print("Preflight PASSED — immediately promoting to Stage 2 training!\n", flush=True)

cmd = [
    sys.executable, "-m", "scale.pretrain",
    *common_args,
    "--dataset", DATASET,
    "--total-tokens", str(TOTAL_TOKENS),
    "--phase", PHASE,
    "--warmup-steps", str(WARMUP_STEPS),
    "--decay-steps", str(DECAY_STEPS),
    "--max-hours", str(MAX_HOURS),
    "--save-every", str(SAVE_EVERY),
    "--eval-every", str(EVAL_EVERY),
    "--sample-every", str(SAMPLE_EVERY),
    "--log-every", str(LOG_EVERY),
    "--keep-last-checkpoints", str(KEEP_LAST_CHECKPOINTS),
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--shard-cache-dir", SHARD_CACHE_DIR,
    "--sync-backend", SYNC_BACKEND,
]
if MAX_STEPS is not None:
    cmd += ["--max-steps", str(MAX_STEPS)]
if REMOTE_TARGET:
    cmd += ["--remote-target", str(REMOTE_TARGET)]

print("Launching:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Pretraining exited with code {rc}")


## 4. Plot Training Loss & `val_bpb` Curve


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

metrics_file = Path(CHECKPOINT_DIR) / "metrics.jsonl"
if metrics_file.exists():
    rows = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
    train_rows = [r for r in rows if "loss" in r]
    eval_rows = [r for r in rows if "val_bpb" in r]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=130)
    if train_rows:
        axes[0].plot([r["tokens_seen"] / 1e9 for r in train_rows], [r["loss"] for r in train_rows], color="#38bdf8")
        axes[0].set_title("1b-ve Training Loss vs Tokens (B)")
        axes[0].set_xlabel("Tokens Seen (Billions)")
        axes[0].set_ylabel("Cross-Entropy Loss (nats)")
        axes[0].grid(alpha=0.3)
    if eval_rows:
        axes[1].plot([r["tokens_seen"] / 1e9 for r in eval_rows], [r["val_bpb"] for r in eval_rows], marker="o", color="#22c55e")
        axes[1].set_title("1b-ve Validation val_bpb vs Tokens (B)")
        axes[1].set_xlabel("Tokens Seen (Billions)")
        axes[1].set_ylabel("val_bpb (lower is better)")
        axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No metrics.jsonl yet — run Cell 3 first.")
